In [7]:
!mkdir assets

mkdir: cannot create directory ‘assets’: File exists


In [1]:
!wget -P 'assets' 'https://raw.githubusercontent.com/modernpacifist/engineering-of-machine-learning-urfu/refs/heads/master/data-storing-and-management/hw5/assets/Invoice-set1.json'

--2025-05-13 14:53:47--  https://raw.githubusercontent.com/modernpacifist/engineering-of-machine-learning-urfu/refs/heads/master/data-storing-and-management/hw5/assets/Invoice-set1.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 353235 (345K) [text/plain]
Saving to: ‘Invoice-set1.json’

Invoice-set1.json   100%[===================>] 344.96K  --.-KB/s    in 0.05s   

2025-05-13 14:53:47 (7.49 MB/s) - ‘Invoice-set1.json’ saved [353235/353235]



In [2]:
!wget -P 'assets' 'https://raw.githubusercontent.com/modernpacifist/engineering-of-machine-learning-urfu/refs/heads/master/data-storing-and-management/hw5/assets/Invoice-set2.json'

--2025-05-13 14:53:58--  https://raw.githubusercontent.com/modernpacifist/engineering-of-machine-learning-urfu/refs/heads/master/data-storing-and-management/hw5/assets/Invoice-set2.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 344396 (336K) [text/plain]
Saving to: ‘Invoice-set2.json’

Invoice-set2.json   100%[===================>] 336.32K  --.-KB/s    in 0.05s   

2025-05-13 14:53:59 (6.87 MB/s) - ‘Invoice-set2.json’ saved [344396/344396]



In [3]:
!wget -P 'assets' 'https://raw.githubusercontent.com/modernpacifist/engineering-of-machine-learning-urfu/refs/heads/master/data-storing-and-management/hw5/assets/Invoice-set3.json'

--2025-05-13 14:54:07--  https://raw.githubusercontent.com/modernpacifist/engineering-of-machine-learning-urfu/refs/heads/master/data-storing-and-management/hw5/assets/Invoice-set3.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 412765 (403K) [text/plain]
Saving to: ‘Invoice-set3.json’

Invoice-set3.json   100%[===================>] 403.09K  --.-KB/s    in 0.05s   

2025-05-13 14:54:07 (7.25 MB/s) - ‘Invoice-set3.json’ saved [412765/412765]



# Задание 1. Structured Streaming из набора файлов

In [4]:
# Import necessary libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, col, from_json, to_json, struct
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, MapType, FloatType
import os
import time

In [5]:
# Initialize Spark Session with necessary configurations
spark = SparkSession.builder \
    .appName("Structured Streaming from Files") \
    .config("spark.sql.streaming.checkpointLocation", "checkpoint") \
    .getOrCreate()

# Set log level to reduce verbosity
spark.sparkContext.setLogLevel("WARN")

In [8]:
# Define the path to the assets directory
assets_dir = "assets"

# Create checkpoint directory if it doesn't exist
checkpoint_dir = "checkpoint"
if not os.path.exists(checkpoint_dir):
    os.makedirs(checkpoint_dir)

# Create output directory if it doesn't exist
output_dir = "output"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Check that the assets directory exists
if not os.path.exists(assets_dir):
    print(f"Warning: The assets directory '{assets_dir}' doesn't exist!")
    print("Please make sure the directory exists and contains JSON files.")
else:
    print(f"Assets directory found at '{assets_dir}'")
    # List JSON files in the directory
    json_files = [f for f in os.listdir(assets_dir) if f.endswith('.json')]
    print(f"Found {len(json_files)} JSON files: {json_files[:5] if len(json_files) > 5 else json_files}")

Assets directory found at 'assets'
Found 3 JSON files: ['Invoice-set2.json', 'Invoice-set3.json', 'Invoice-set1.json']


# Задание 1.2: Открыть на основе исходных json файлов потоковый датафрейм

In [9]:
# Define the schema for JSON data (adapt this to your actual data structure)
# This is an example schema - you'll need to adjust it based on your actual JSON structure
schema = StructType([
    StructField("id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("user", StructType([
        StructField("id", StringType(), True),
        StructField("name", StringType(), True),
        StructField("email", StringType(), True)
    ]), True),
    StructField("items", ArrayType(StructType([
        StructField("item_id", StringType(), True),
        StructField("name", StringType(), True),
        StructField("price", FloatType(), True),
        StructField("quantity", IntegerType(), True)
    ])), True),
    StructField("metadata", MapType(StringType(), StringType()), True)
])

# Create a streaming DataFrame from the JSON files in the assets directory
streamingDF = spark.readStream \
    .schema(schema) \
    .option("maxFilesPerTrigger", 2) \
    .json(assets_dir)

# Show the schema of the streaming DataFrame
print("Streaming DataFrame Schema:")
streamingDF.printSchema()

Streaming DataFrame Schema:
root
 |-- id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- user: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- email: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- price: float (nullable = true)
 |    |    |-- quantity: integer (nullable = true)
 |-- metadata: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)



# Задание 1.3: Написать преобразование исходного датафрейма для получения плоской структуры

In [10]:
# Flatten the nested structure
# This flattening approach will depend on your actual data schema
def flatten_df(nested_df):
    # Flatten user fields
    flattened_df = nested_df \
        .withColumn("user_id", col("user.id")) \
        .withColumn("user_name", col("user.name")) \
        .withColumn("user_email", col("user.email"))

    # Explode the items array
    flattened_df = flattened_df \
        .withColumn("item", explode("items"))

    # Flatten item fields
    flattened_df = flattened_df \
        .withColumn("item_id", col("item.item_id")) \
        .withColumn("item_name", col("item.name")) \
        .withColumn("item_price", col("item.price")) \
        .withColumn("item_quantity", col("item.quantity"))

    # Drop nested columns
    flattened_df = flattened_df \
        .drop("user", "items", "item")

    # For MapType columns like metadata, you can either:
    # 1. Convert it to JSON string
    flattened_df = flattened_df \
        .withColumn("metadata_json", to_json(col("metadata")))

    # 2. Or explode it into separate columns if you know specific keys
    # Example:
    # .withColumn("metadata_source", col("metadata.source"))
    # .withColumn("metadata_version", col("metadata.version"))

    # Drop the original metadata map
    flattened_df = flattened_df.drop("metadata")

    return flattened_df

# Apply flattening transformation
flattenedDF = flatten_df(streamingDF)

# Show the schema of the flattened DataFrame
print("Flattened DataFrame Schema:")
flattenedDF.printSchema()

Flattened DataFrame Schema:
root
 |-- id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- user_name: string (nullable = true)
 |-- user_email: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- item_name: string (nullable = true)
 |-- item_price: float (nullable = true)
 |-- item_quantity: integer (nullable = true)
 |-- metadata_json: string (nullable = true)



# Задание 1.4: Сохранять получаемые каждые X секунд исходные файлы в json

In [11]:
# Define trigger interval in seconds
trigger_interval = 10  # You can adjust this value

# Write the flattened DataFrame to JSON files
query = flattenedDF.writeStream \
    .format("json") \
    .option("path", output_dir) \
    .option("checkpointLocation", checkpoint_dir) \
    .trigger(processingTime=f"{trigger_interval} seconds") \
    .start()

# Print information about the streaming query
print(f"Streaming query started. Writing to {output_dir} every {trigger_interval} seconds.")
print("Query ID:", query.id)
print("Query Name:", query.name)

# Wait for the streaming query to terminate
try:
    print("Streaming is active. Press Ctrl+C to terminate...")
    query.awaitTermination()
except KeyboardInterrupt:
    print("Streaming query terminated by user.")
    query.stop()

Streaming query started. Writing to output every 10 seconds.
Query ID: 43f228d5-86d2-4f12-8445-06d3e8ed33a5
Query Name: None
Streaming is active. Press Ctrl+C to terminate...


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


Streaming query terminated by user.


# Задание 2. Применение Tumbling Window

In [12]:
# Import necessary libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import window, col, count, avg, max, min, expr, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, TimestampType
import os
import time
import json
import random
from datetime import datetime, timedelta

In [13]:
# Initialize Spark Session with necessary configurations
spark = SparkSession.builder \
    .appName("Tumbling Window Streaming") \
    .config("spark.sql.streaming.checkpointLocation", "checkpoint_window") \
    .getOrCreate()

# Set log level to reduce verbosity
spark.sparkContext.setLogLevel("WARN")

# Задание 2.1: Подготовить подходящие данные

In [14]:
# Create data directories if they don't exist
input_dir = "streaming_input"
output_dir = "window_output"
checkpoint_dir = "checkpoint_window"

for directory in [input_dir, output_dir, checkpoint_dir]:
    if not os.path.exists(directory):
        os.makedirs(directory)

# Define a data generator for time-series data (simulating IoT sensor readings)
def generate_sensor_data(num_records=100, start_time=None):
    """Generate time-series sensor data with timestamps"""
    sensor_data = []

    if start_time is None:
        # Start from current time minus 1 hour
        start_time = datetime.now() - timedelta(hours=1)

    # Define some sensor types
    sensor_types = ["temperature", "humidity", "pressure", "co2"]
    locations = ["room1", "room2", "kitchen", "livingroom", "bedroom"]

    for i in range(num_records):
        # Generate timestamp with some random progression
        event_time = start_time + timedelta(seconds=i*random.randint(5, 15))

        # Create a record
        record = {
            "sensor_id": f"sensor_{random.randint(1, 10)}",
            "sensor_type": random.choice(sensor_types),
            "location": random.choice(locations),
            "value": round(random.uniform(15, 35), 2),  # Random value between 15 and 35
            "timestamp": event_time.strftime("%Y-%m-%d %H:%M:%S")
        }
        sensor_data.append(record)

    return sensor_data

# Generate initial batch of data
for batch_id in range(5):
    # Generate data with timestamps roughly 10 minutes apart between batches
    start_time = datetime.now() - timedelta(hours=1, minutes=50-batch_id*10)
    data = generate_sensor_data(num_records=20, start_time=start_time)

    # Write to JSON file
    filename = f"{input_dir}/sensors_batch_{batch_id}.json"
    with open(filename, 'w') as f:
        for record in data:
            f.write(json.dumps(record) + '\n')

    print(f"Generated batch {batch_id} with 20 records, saved to {filename}")

# Define function to generate more data while streaming is active
def generate_more_data(directory, interval_seconds=15, num_batches=10):
    """Generate additional data files at regular intervals"""
    for batch_id in range(100, 100 + num_batches):
        # Generate data with current timestamp
        data = generate_sensor_data(num_records=10)

        # Write to JSON file
        filename = f"{directory}/sensors_batch_{batch_id}.json"
        with open(filename, 'w') as f:
            for record in data:
                f.write(json.dumps(record) + '\n')

        print(f"Generated additional batch {batch_id} with 10 records")

        # Wait for the interval
        time.sleep(interval_seconds)

Generated batch 0 with 20 records, saved to streaming_input/sensors_batch_0.json
Generated batch 1 with 20 records, saved to streaming_input/sensors_batch_1.json
Generated batch 2 with 20 records, saved to streaming_input/sensors_batch_2.json
Generated batch 3 with 20 records, saved to streaming_input/sensors_batch_3.json
Generated batch 4 with 20 records, saved to streaming_input/sensors_batch_4.json


# Задание 2.2: Открыть на основе исходных json файлов потоковый датафрейм

In [15]:
# Define the schema for sensor data
schema = StructType([
    StructField("sensor_id", StringType(), True),
    StructField("sensor_type", StringType(), True),
    StructField("location", StringType(), True),
    StructField("value", FloatType(), True),
    StructField("timestamp", StringType(), True)
])

# Create a streaming DataFrame from the JSON files
streamingDF = spark.readStream \
    .schema(schema) \
    .option("maxFilesPerTrigger", 2) \
    .json(input_dir)

# Convert string timestamp to timestamp type for windowing operations
streamingDF = streamingDF.withColumn(
    "event_timestamp",
    to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss")
)

# Show the schema of the streaming DataFrame
print("Streaming DataFrame Schema:")
streamingDF.printSchema()

Streaming DataFrame Schema:
root
 |-- sensor_id: string (nullable = true)
 |-- sensor_type: string (nullable = true)
 |-- location: string (nullable = true)
 |-- value: float (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)



# Задание 2.3: Написать произвольный запрос агрегации, используя функцию tumbling window

In [16]:
# Define the tumbling window duration
window_duration = "5 minutes"

# Apply tumbling window aggregation
# Here we'll calculate average, min, and max values per sensor type and location in each window
windowedAggregation = streamingDF \
    .groupBy(
        window(col("event_timestamp"), window_duration),
        col("sensor_type"),
        col("location")
    ) \
    .agg(
        count("*").alias("count"),
        avg("value").alias("avg_value"),
        min("value").alias("min_value"),
        max("value").alias("max_value")
    ) \
    .orderBy(col("window.start").desc())

# Format the window start and end times as separate columns for better readability in output
windowedAggregation = windowedAggregation \
    .withColumn("window_start", col("window.start").cast("string")) \
    .withColumn("window_end", col("window.end").cast("string")) \
    .drop("window")

# Show the schema of the windowed aggregation DataFrame
print("Windowed Aggregation DataFrame Schema:")
windowedAggregation.printSchema()

Windowed Aggregation DataFrame Schema:
root
 |-- sensor_type: string (nullable = true)
 |-- location: string (nullable = true)
 |-- count: long (nullable = false)
 |-- avg_value: double (nullable = true)
 |-- min_value: float (nullable = true)
 |-- max_value: float (nullable = true)
 |-- window_start: string (nullable = true)
 |-- window_end: string (nullable = true)



# Задание 2.4: Сохранять получаемые каждые X секунд исходные файлы в json

In [20]:
# Задание 4: Сохранять получаемые каждые X секунд исходные файлы в json

# Define trigger interval in seconds
trigger_interval = 15  # Process every 15 seconds

# For windowed aggregations, we need to either:
# 1. Use "update" mode with JSON format
# 2. Or add a watermark and use "append" mode

# Add watermark to allow for late data and enable append mode for windowed aggregation
windowedAggregation = streamingDF \
    .withWatermark("event_timestamp", "10 minutes") \
    .groupBy(
        window(col("event_timestamp"), window_duration),
        col("sensor_type"),
        col("location")
    ) \
    .agg(
        count("*").alias("count"),
        avg("value").alias("avg_value"),
        min("value").alias("min_value"),
        max("value").alias("max_value")
    ) \
    .withColumn("window_start", col("window.start").cast("string")) \
    .withColumn("window_end", col("window.end").cast("string")) \
    .drop("window")

# Write the windowed aggregation to JSON files
query = windowedAggregation.writeStream \
    .format("json") \
    .option("path", output_dir) \
    .option("checkpointLocation", checkpoint_dir) \
    .trigger(processingTime=f"{trigger_interval} seconds") \
    .outputMode("append") \
    .start()

print(f"Streaming query started. Writing to {output_dir} every {trigger_interval} seconds.")
print("Query ID:", query.id)
print("Query Name:", query.name)

# Generate additional data while streaming is active
print("Starting to generate additional data...")

Streaming query started. Writing to window_output every 15 seconds.
Query ID: ae3a204e-307d-46ca-828a-1feb4d02a96a
Query Name: None
Starting to generate additional data...


# Задание 3. Применение Sliding Window

In [21]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import window, col, count, avg, max, min, expr, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, TimestampType
import os
import time
import json
import random
from datetime import datetime, timedelta

In [22]:
# Initialize Spark Session with necessary configurations
spark = SparkSession.builder \
    .appName("Sliding Window Streaming") \
    .config("spark.sql.streaming.checkpointLocation", "checkpoint_sliding") \
    .getOrCreate()

# Set log level to reduce verbosity
spark.sparkContext.setLogLevel("WARN")

# Задание 1: Подготовить подходящие данные

In [39]:
# Create data directories if they don't exist
input_dir = "streaming_input_sliding"
output_dir = "window_output_sliding"
checkpoint_dir = "checkpoint_sliding"

for directory in [input_dir, output_dir, checkpoint_dir]:
    if not os.path.exists(directory):
        os.makedirs(directory)

# Complete revised data generator with strict type handling
def generate_stock_data(num_records=100, start_time=None):
    """Generate time-series stock price data with timestamps"""
    stock_data = []

    if start_time is None:
        # Start from current time minus 1 hour
        start_time = datetime.now() - timedelta(hours=1)

    # Define some stock symbols
    stock_symbols = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA"]
    sectors = ["Technology", "Finance", "Healthcare", "Consumer", "Energy"]

    # Starting prices for each stock
    base_prices = {
        "AAPL": 150.0,
        "MSFT": 280.0,
        "GOOGL": 120.0,
        "AMZN": 130.0,
        "TSLA": 250.0
    }

    # For each record, generate a timestamp and stock data
    for i in range(int(num_records)):  # Ensure integer
        # Generate timestamp with some random progression
        seconds_offset = i * random.randint(2, 8)
        event_time = start_time + timedelta(seconds=seconds_offset)

        # Select random stock
        symbol = random.choice(stock_symbols)
        base_price = float(base_prices[symbol])  # Ensure float

        # Generate price with some random movement
        price_change = float(random.uniform(-5.0, 5.0))  # Ensure float
        price = base_price + price_change

        # Update base price for next time this stock is selected
        base_prices[symbol] = price

        # Generate volume as integer
        volume = int(random.randint(100, 10000))  # Ensure integer

        # Create a record with explicit types
        record = {
            "symbol": str(symbol),
            "sector": str(random.choice(sectors)),
            "price": float(round(price, 2)),
            "volume": int(volume),
            "timestamp": str(event_time.strftime("%Y-%m-%d %H:%M:%S"))
        }
        stock_data.append(record)

    return stock_data

# Generate initial batch of data
for batch_id in range(5):
    # Generate data with timestamps roughly 5 minutes apart between batches
    start_time = datetime.now() - timedelta(hours=1, minutes=25-batch_id*5)
    data = generate_stock_data(num_records=30, start_time=start_time)

    # Write to JSON file
    filename = f"{input_dir}/stocks_batch_{batch_id}.json"
    with open(filename, 'w') as f:
        for record in data:
            f.write(json.dumps(record) + '\n')

    print(f"Generated batch {batch_id} with 30 records, saved to {filename}")

Generated batch 0 with 30 records, saved to streaming_input_sliding/stocks_batch_0.json
Generated batch 1 with 30 records, saved to streaming_input_sliding/stocks_batch_1.json
Generated batch 2 with 30 records, saved to streaming_input_sliding/stocks_batch_2.json
Generated batch 3 with 30 records, saved to streaming_input_sliding/stocks_batch_3.json
Generated batch 4 with 30 records, saved to streaming_input_sliding/stocks_batch_4.json


# Задание 2: Открыть на основе исходных json файлов потоковый датафрейм

In [40]:
# Define the schema for stock data
schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("sector", StringType(), True),
    StructField("price", FloatType(), True),
    StructField("volume", IntegerType(), True),
    StructField("timestamp", StringType(), True)
])

# Create a streaming DataFrame from the JSON files
streamingDF = spark.readStream \
    .schema(schema) \
    .option("maxFilesPerTrigger", 2) \
    .json(input_dir)

# Convert string timestamp to timestamp type for windowing operations
streamingDF = streamingDF.withColumn(
    "event_timestamp",
    to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss")
)

# Show the schema of the streaming DataFrame
print("Streaming DataFrame Schema:")
streamingDF.printSchema()

Streaming DataFrame Schema:
root
 |-- symbol: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- price: float (nullable = true)
 |-- volume: integer (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)



# Задание 3: Написать произвольный запрос агрегации, используя функцию sliding window

In [43]:
# Define sliding window parameters
window_duration = "10 minutes"  # Window size
slide_duration = "2 minutes"    # How often a new window starts

# First, ensure price and volume columns are proper numeric types
streamingDF = streamingDF \
    .withColumn("price", col("price").cast("float")) \
    .withColumn("volume", col("volume").cast("integer"))

# Apply sliding window aggregation with explicit casting
slidingWindowAggregation = streamingDF \
    .withWatermark("event_timestamp", "15 minutes") \
    .groupBy(
        window(col("event_timestamp"), window_duration, slide_duration),
        col("symbol"),
        col("sector")
    ) \
    .agg(
        count("*").alias("num_trades"),
        avg("price").cast("float").alias("avg_price"),
        min("price").cast("float").alias("min_price"),
        max("price").cast("float").alias("max_price"),
        (max("price").cast("float") - min("price").cast("float")).alias("price_range"),
    ) \
    .withColumn("price_volatility",
                (col("price_range") / col("avg_price") * 100).cast("float")) \
    .withColumn("window_start", col("window.start").cast("string")) \
    .withColumn("window_end", col("window.end").cast("string")) \
    .drop("window", "price_range")

# Show the schema of the windowed aggregation DataFrame
print("Sliding Window Aggregation DataFrame Schema:")
slidingWindowAggregation.printSchema()

Sliding Window Aggregation DataFrame Schema:
root
 |-- symbol: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- num_trades: long (nullable = false)
 |-- avg_price: float (nullable = true)
 |-- min_price: float (nullable = true)
 |-- max_price: float (nullable = true)
 |-- price_volatility: float (nullable = true)
 |-- window_start: string (nullable = true)
 |-- window_end: string (nullable = true)



# Задание 4: Сохранять получаемые каждые X секунд исходные файлы в json

In [44]:
# Define trigger interval in seconds
trigger_interval = 15  # Process every 15 seconds

# Write the windowed aggregation to JSON files
query = slidingWindowAggregation.writeStream \
    .format("json") \
    .option("path", output_dir) \
    .option("checkpointLocation", checkpoint_dir) \
    .trigger(processingTime=f"{trigger_interval} seconds") \
    .outputMode("append") \
    .start()

print(f"Streaming query started. Writing to {output_dir} every {trigger_interval} seconds.")
print("Query ID:", query.id)
print("Query Name:", query.name)

# Generate additional data while streaming is active
print("Starting to generate additional data...")

Streaming query started. Writing to window_output_sliding every 15 seconds.
Query ID: b69edf91-44c4-4408-8fe8-2a1923ab21e2
Query Name: None
Starting to generate additional data...


In [45]:
# Run streaming for a fixed amount of time, generating new data in parallel
try:
    # Start generating more data
    for batch_id in range(100, 110):
        # Generate data with timestamps around current time
        data = generate_stock_data(num_records=15)

        # Write to JSON file
        filename = f"{input_dir}/stocks_batch_{batch_id}.json"
        with open(filename, 'w') as f:
            for record in data:
                f.write(json.dumps(record) + '\n')

        print(f"Generated additional batch {batch_id} with 15 records")

        # Wait a bit before the next batch
        time.sleep(trigger_interval)

        # Print current aggregation metrics if available
        if query.lastProgress:
            print(f"Last processed batch had {query.lastProgress['numInputRows']} rows")

    # Wait for final processing
    print("Waiting for final processing...")
    time.sleep(trigger_interval * 2)

except KeyboardInterrupt:
    print("Streaming terminated by user.")
finally:
    query.stop()
    print("Streaming stopped.")

Generated additional batch 100 with 15 records
Last processed batch had 60 rows
Generated additional batch 101 with 15 records
Streaming terminated by user.
Streaming stopped.


In [47]:
# Stop Spark session
spark.stop()